# Lab 10 — Testing Endpoints with dependency_overrides

**Difficulty: Intermediate | ~40 min | Requires Lab 4 (Dependency Injection)**

### Step 0: Install Dependencies

This cell installs every pinned dependency the lab needs. Run this first so all later cells have what they require.

In [1]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports, API Key, and App Setup

Standard imports for the app. `Depends` is how FastAPI injects dependencies into endpoints — you declare what you need, and FastAPI figures out how to provide it. `TestClient` (re-exported by FastAPI from Starlette) lets us make HTTP requests directly in-process without starting a real server, which keeps tests fast and deterministic. The OpenRouter client is set up the same way as in Lab 9.

In [2]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import AsyncOpenAI
import os

load_dotenv()
api_key = os.getenv("OPEN_ROUTER_KEY") or input("Open Router API key: ")
client = AsyncOpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
app = FastAPI()
test_client = TestClient(app)

### Step 2: classify_message — The Production Classifier

A single async function that makes a real LLM call asking the model to respond with exactly "safe" or "unsafe". Because it's a plain function, the dependency function (in Step 3) can return it directly, and the endpoint can call it as `classifier(message)`.

In [3]:
async def classify_message(message):
    response = await client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": (
            f"Classify this message as exactly one word: 'safe' or 'unsafe'. "
            f"Message: {message}"
        )}],
    )
    return response.choices[0].message.content.strip().lower()

### Step 3: get_classifier_client — The Classifier Dependency

This is the dependency function that FastAPI will inject into the endpoint. It returns the `classify_message` function itself. During tests, `app.dependency_overrides[get_classifier_client]` replaces this with a lambda that returns a fake instead.

In [4]:
async def get_classifier_client():
    return classify_message

### Step 4: generate_reply — The Production Replier

A separate function that makes a real LLM call to produce the actual reply text. Having the replier as its own dependency — separate from the classifier — is what lets the tests prove the replier was or was not called independently.

In [5]:
async def generate_reply(message):
    response = await client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": f"Generate a polite reply to: {message}"}],
    )
    return response.choices[0].message.content

### Step 5: get_reply_client — The Replier Dependency

The dependency function for the replier, following the same pattern as the classifier.

In [6]:
async def get_reply_client():
    return generate_reply

### Step 6: The POST /submit Endpoint

This is the core of the lab. The endpoint receives a message, calls the classifier via its injected dependency, and branches on the result:

- **`"safe"`** → call the replier and return the approved reply
- **`"unsafe"`** → reject immediately, never call the replier
- **anything else** → fail closed (reject, never call the replier)
- **exception** → return a clean error, don't crash

The dependency-injected `classifier` and `replier` are each just a callable function, so the endpoint invokes them directly: `classifier(req.message)` and `replier(req.message)`. The `try/except` around `classifier(req.message)` ensures an unhandled LLM error becomes a structured JSON response, not a 500 crash.

In [7]:
class MessageRequest(BaseModel):
    message: str

@app.post("/submit")
async def submit(
    req: MessageRequest,
    classifier=Depends(get_classifier_client),
    replier=Depends(get_reply_client),
):
    try:
        classification = await classifier(req.message)
    except Exception as exc:
        return {"status": "error", "detail": str(exc)}

    if classification == "unsafe":
        return {"status": "rejected", "reason": "unsafe"}
    if classification != "safe":
        return {"status": "rejected", "reason": "unrecognized classification"}

    reply = await replier(req.message)
    return {"status": "approved", "reply": reply}

### Step 7: make_fake_classifier — The Fake Classifier

A factory that returns a "fake classifier": an async function that just returns a fixed string (or raises an exception if configured to). Python functions can carry attributes, so the returned function tracks its own `call_count` — incremented on every call, which lets us prove how many times the classifier was invoked.

In [8]:
def make_fake_classifier(response=None, raise_error=False):

    async def fake_classify(message):
        fake_classify.call_count += 1
        if raise_error:
            raise RuntimeError("Classifier service unavailable")
        return response
        
    fake_classify.call_count = 0
    return fake_classify

### Step 8: make_fake_replier — The Fake Replier

Same pattern: a factory returning a plain async function with a fixed return value and a `call_count` attribute that increments on every call. This is what makes it possible to prove the replier was or was not invoked — not just infer it from the response shape.

In [9]:
def make_fake_replier(response="fake reply"):

    async def fake_generate(message):
        fake_generate.call_count += 1
        return response
        
    fake_generate.call_count = 0
    return fake_generate

### Test 1: Safe Message Gets Approved and Replied

This test overrides both dependencies: the classifier fake returns `"safe"` and the replier fake returns a canned reply. The assertions check three things: the status is `"approved"`, the reply matches exactly, and `call_count == 1` on the replier — proving the second call genuinely fired, not just that the response happened to look right.

In [ ]:
def test_safe_message_gets_approved_and_replied():
    fake_replier = make_fake_replier(response="Hello! How can I help you today?")
    fake_classifier = make_fake_classifier(response="safe")
    
    app.dependency_overrides[get_classifier_client] = lambda: fake_classifier
    app.dependency_overrides[get_reply_client] = lambda: fake_replier

    response = test_client.post("/submit", json={"message": "Hello there"})
    data = response.json()

    assert data["status"] == "approved"
    assert data["reply"] == "Hello! How can I help you today?"
    assert fake_replier.call_count == 1
    print("Test 1: safe → approved")
    print("PASSED")

After every test, `app.dependency_overrides.clear()` removes all overrides so the next test starts clean. Without clearing, one test's fake could silently leak into the next test and produce a misleading result.

In [19]:
test_safe_message_gets_approved_and_replied()
app.dependency_overrides.clear()

Test 1: safe → approved
PASSED


### Test 2: Unsafe Message Rejected Without Reply Call

This is the important proof-of-skip test. The classifier fake returns `"unsafe"`, and we assert that the replier's `call_count` is 0 — proving the costlier second call was correctly skipped. You cannot reliably verify this against a real, non-deterministic classifier on a schedule.

In [20]:
def test_unsafe_message_rejected_without_reply_call():
    fake_replier = make_fake_replier(response="This should never appear")
    fake_classifier = make_fake_classifier(response="unsafe")

    app.dependency_overrides[get_classifier_client] = lambda: fake_classifier
    app.dependency_overrides[get_reply_client] = lambda: fake_replier

    response = test_client.post("/submit", json={"message": "Dangerous content"})
    data = response.json()

    assert data["status"] == "rejected"
    assert data["reason"] == "unsafe"
    assert fake_replier.call_count == 0
    print("Test 2: unsafe → rejected, replier not called")
    print("PASSED")

In [21]:
test_unsafe_message_rejected_without_reply_call()
app.dependency_overrides.clear()

Test 2: unsafe → rejected, replier not called
PASSED


### Test 3: Malformed Classification Fails Closed

This is the standout case in the whole lab. You cannot reliably coax a real model into producing a specific malformed answer like `"maybe"` on demand to verify your fail-closed logic actually works — but a fake makes this trivial and instant. The classifier fake returns `"maybe"` (neither `"safe"` nor `"unsafe"`), and we assert the message is rejected with `"unrecognized classification"` and the replier was never called.

In [22]:
def test_malformed_classification_fails_closed():
    fake_replier = make_fake_replier(response="This should never appear")
    fake_classifier = make_fake_classifier(response="maybe")

    app.dependency_overrides[get_classifier_client] = lambda: fake_classifier
    app.dependency_overrides[get_reply_client] = lambda: fake_replier

    response = test_client.post("/submit", json={"message": "Ambiguous content"})
    data = response.json()

    assert data["status"] == "rejected"
    assert data["reason"] == "unrecognized classification"
    assert fake_replier.call_count == 0
    print("Test 3: malformed → rejected (fail closed), replier not called")
    print("PASSED")

In [23]:
test_malformed_classification_fails_closed()
app.dependency_overrides.clear()

Test 3: malformed → rejected (fail closed), replier not called
PASSED


### Test 4: Classifier Failure Returns Clean Error

When the classifier raises an exception (simulating a service outage), the endpoint's `try/except` catches it and returns a structured error response instead of crashing with a 500. We verify the response has `"status": "error"` and a `"detail"` field containing the error message.

In [26]:
def test_classifier_failure_returns_clean_error():
    fake_classifier = make_fake_classifier(raise_error=True)
    app.dependency_overrides[get_classifier_client] = lambda: fake_classifier

    response = test_client.post("/submit", json={"message": "Any message"})
    data = response.json()

    assert data["status"] == "error"
    assert "detail" in data
    assert "Classifier service unavailable" in data["detail"]
    print("Test 4: classifier error → clean error response")
    print("PASSED")

In [27]:
test_classifier_failure_returns_clean_error()
app.dependency_overrides.clear()

print("\nAll four tests passed.")

Test 4: classifier error → clean error response
PASSED

All four tests passed.
